In [9]:
import pandas as pd
from functools import reduce
 
files = [
    'data/crypto_market_data_daily.csv',
    'data/google_trends_data.csv',
    'data/macro_data.csv',
    'data/sentiment_data.csv'
]
 
dfs = []
for f in files:
    df = pd.read_csv(f)
    df = df.rename(columns={'timestamp': 'date'}).drop_duplicates('date').dropna(subset=['date'])
    dfs.append(df)

merged = reduce(lambda l, r: pd.merge(l, r, on='date', how='outer'), dfs)
merged = merged.sort_values('date').reset_index(drop=True)
merged.to_csv('integrated_data.csv', index=False)
print(f"Rows: {len(merged)} | Cols: {len(merged.columns)}")

Rows: 3330 | Cols: 20


In [ ]:
# Clean merged
merged = merged.copy()
merged['date'] = pd.to_datetime(merged['date'], errors='coerce')

text_cols = [c for c in ['symbol', 'value_classification'] if c in merged.columns]
for c in text_cols:
    merged[c] = merged[c].astype('string').str.strip()

for c in [col for col in merged.columns if col not in ['date', *text_cols]]:
    merged[c] = pd.to_numeric(merged[c], errors='coerce')

merged = (
    merged.dropna(subset=['date'])
          .drop_duplicates(subset='date', keep='first')
          .sort_values('date')
          .reset_index(drop=True)
)

# Drop rows where all non-date fields are missing
value_cols = [c for c in merged.columns if c != 'date']
merged = merged.dropna(how='all', subset=value_cols).reset_index(drop=True)

# See structure
print(f"Cleaned shape: {merged.shape}")
merged.info()
print("\nMissing values (top 10):")
print(merged.isna().sum().sort_values(ascending=False).head(10))
print("\nPreview:")
print(merged.head())